In [1]:
# ============================================================
# Cell 1 — Setup
# Works in Colab or from a local checkout of the repository.
# ============================================================

import os
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
BRANCH    = "Manuel"  # change if running another branch in Colab

if IN_COLAB:
    REPO_ROOT = "/content/capstone-quantum"
    %cd /content
    !rm -rf capstone-quantum
    !git clone {REPO_URL}
    %cd /content/capstone-quantum
    !git checkout {BRANCH}
else:
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    REPO_ROOT = str(root)
    %cd {REPO_ROOT}

# verify expected directories exist
for d in ["data/processed", "notebooks", "results", "src"]:
    path = os.path.join(REPO_ROOT, d)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {d:<25} {status}")

# create results directory if it doesn't exist yet
os.makedirs(os.path.join(REPO_ROOT, "results"), exist_ok=True)

print("\nSetup complete.")


  data/processed            OK
  notebooks                 OK
  results                   OK
  src                       OK

Setup complete.


In [2]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import sys
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
)

warnings.filterwarnings("ignore")

# add repo root to path so src/ is importable
sys.path.insert(0, REPO_ROOT)

print("Imports complete.")

Imports complete.


In [3]:
# ============================================================
# Cell 3 — Configuration
# ============================================================

# --- Paths ---
DATA_DIR    = Path(REPO_ROOT) / "data" / "processed"
RESULTS_DIR = Path(REPO_ROOT) / "results"

# --- Window Parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small integer for a quick smoke test

# --- Evaluation Parameters ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
ACTIVE_TRACKS = ["B"]  # Track-B multi-file notebook

# --- Track Registry ---
# Track B files follow Jack's requested exact format:
#   f"track_b_{target_name}.parquet"
TRACK_CONFIG = {
    "A":      {target: {"type": "dataframe", "path": DATA_DIR / "track_A_full.parquet"} for target in TARGET_LIST},
    "B":      {target: {"type": "dataframe", "path": DATA_DIR / f"track_b_{target}.parquet"} for target in TARGET_LIST},
    "C":      {target: {"type": "pipeline",  "path": DATA_DIR / "track_C_pipe.pkl"} for target in TARGET_LIST},  # tabled
    "D":      {target: {"type": "pipeline",  "path": DATA_DIR / "track_D_pipe.pkl"} for target in TARGET_LIST},
    "D_mini": {target: {"type": "pipeline",  "path": DATA_DIR / "track_D_mini_pipe.pkl"} for target in TARGET_LIST},
}

# --- Target Column Map ---
# maps (target, horizon) -> column name in targets.parquet
TARGET_COL_MAP = {
    (target, h): f"y_{target}_h{h}"
    for target in TARGET_LIST
    for h in HORIZON_LIST
}

print("Configuration set.")
print(f"  Window:  {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"  Targets: {TARGET_LIST}")
print(f"  Horizons: {HORIZON_LIST}")
print(f"  Active tracks: {ACTIVE_TRACKS}")


Configuration set.
  Window:  120m train / 1m test / 1m step
  Targets: ['INDPRO', 'PAYEMS']
  Horizons: [1, 3, 6]
  Active tracks: ['B']


In [4]:
# ============================================================
# Cell 4 — Data Loading
# ============================================================

# --- Load track DataFrames ---
track_data = {}

for track_name, dict in TRACK_CONFIG.items():
    if track_name not in ACTIVE_TRACKS:
            print(f"  Track {track_name:<8} SKIPPED (inactive)")
            continue
    else:
        track_data[track_name] = {} 
        for target_name, config in dict.items():
            if config["type"] == "dataframe":
                track_data[track_name][target_name] = pd.read_parquet(config["path"])
                print(f"  Track {track_name:<8}, target {target_name:<8} loaded   shape={track_data[track_name][target_name].shape}")
            elif config["type"] == "pipeline":
                with open(config["path"], "rb") as f:
                    track_data[track_name][target_name] = pickle.load(f)
                print(f"  Track {track_name:<8}, target {target_name:<8} loaded   pipeline={type(track_data[track_name][target_name]).__name__}")

# --- Load stationary panel (needed for pipeline tracks) ---
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
print(f"\n  Stationary panel loaded   shape={stationary_panel.shape}")

# --- Load targets ---
targets = pd.read_parquet(DATA_DIR / "targets.parquet")
print(f"  Targets loaded            shape={targets.shape}")

# --- Load USREC ---
usrec = pd.read_parquet(DATA_DIR / "usrec.parquet")
print(f"  USREC loaded              shape={usrec.shape}")

print("\nData loading complete.")

  Track A        SKIPPED (inactive)
  Track B       , target INDPRO   loaded   shape=(767, 10)
  Track B       , target PAYEMS   loaded   shape=(767, 9)
  Track B       , target CPIAUCSL loaded   shape=(767, 6)
  Track B       , target S&P 500  loaded   shape=(767, 6)
  Track D        SKIPPED (inactive)
  Track D_mini   SKIPPED (inactive)

  Stationary panel loaded   shape=(767, 123)
  Targets loaded            shape=(806, 20)
  USREC loaded              shape=(806, 1)

Data loading complete.


In [5]:
print("First track start:", track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index.min())
print("First track end:  ", track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index.max())

First track start: 1962-04-01 00:00:00
First track end:   2026-02-01 00:00:00


In [6]:
for name in ACTIVE_TRACKS:
    if isinstance(track_data[name], pd.DataFrame):
        print(f"  Track {name:<8} {track_data[name].index.min()} -> {track_data[name].index.max()}")
print(f"  Stationary   {stationary_panel.index.min()} -> {stationary_panel.index.max()}")

  Stationary   1962-04-01 00:00:00 -> 2026-02-01 00:00:00


In [7]:
def generate_folds(index, window_months, test_months, step_months):
    """
    Generate fixed sliding window pseudo-OOS folds.

    Parameters
    ----------
    index         : pd.DatetimeIndex — full date index of the panel
    window_months : int — training window length in months
    test_months   : int — test window length in months
    step_months   : int — step size between folds in months

    Returns
    -------
    folds : list of dict with train/test date boundaries
    """
    folds = []
    n = len(index)

    for start in range(0, n - window_months - test_months + 1, step_months):
        train_start = index[start]
        train_end   = index[start + window_months - 1]
        test_start  = index[start + window_months]
        test_end    = index[start + window_months + test_months - 1]

        folds.append({
            "fold_id":     len(folds),
            "train_start": train_start,
            "train_end":   train_end,
            "test_start":  test_start,
            "test_end":    test_end,
        })

    return folds

# generate folds using track A index as reference
folds = generate_folds(
    index         = track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index,
    window_months = WINDOW_MONTHS,
    test_months   = TEST_MONTHS,
    step_months   = STEP_MONTHS,
)

if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"  Total folds: {len(folds)}")
print(f"\n  First fold:")
print(f"    train: {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}")
print(f"    test:  {folds[0]['test_start'].date()}  -> {folds[0]['test_end'].date()}")
print(f"\n  Last fold:")
print(f"    train: {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}")
print(f"    test:  {folds[-1]['test_start'].date()}  -> {folds[-1]['test_end'].date()}")


  Total folds: 647

  First fold:
    train: 1962-04-01 -> 1972-03-01
    test:  1972-04-01  -> 1972-04-01

  Last fold:
    train: 2016-02-01 -> 2026-01-01
    test:  2026-02-01  -> 2026-02-01


In [18]:
# ============================================================
# Cell 6 — Metric Functions
# ============================================================

def compute_naive_mae(y_train):
    """
    Lag-1 random-walk MAE on the training target.
    Used as denominator for MASE and RMSSE.
    """
    y = np.asarray(pd.Series(y_train).dropna(), dtype=float)
    if len(y) < 2:
        return np.nan
    return np.mean(np.abs(y[1:] - y[:-1]))


def compute_metrics(y_true, y_pred, y_train):
    """
    Compute metrics using the fold-specific training target as the scale.
    This matches the convention documented in docs/thursday_model_interface.md.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae       = mean_absolute_error(y_true, y_pred)
    mse       = mean_squared_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)

    naive_mae = compute_naive_mae(y_train)

    # avoid division by zero on flat training series
    if not np.isfinite(naive_mae) or naive_mae == 0:
        mase  = np.nan
        rmsse = np.nan
    else:
        mase  = mae / naive_mae
        rmsse = np.sqrt(mse) / naive_mae

    return {
        "mse":       mse,
        "mae":       mae,
        "median_ae": median_ae,
        "mase":      mase,
        "rmsse":     rmsse,
    }

print("Metric functions defined.")


Metric functions defined.


In [19]:
from src.model_functions import (
    HAS_XGB,
    adaboost_di_model_function,
    adaboost_model_function,
    knn_di_model_function,
    random_forest_model_function,
    svr_linear_di_model_function,
    xgboost_di_model_function,
)

# ============================================================
# Cell 7 — Model Registry
# ============================================================

# Jack's latest multi-file Track-B notebook calls every model as:
#   model_fn(train_df, test_df, y_train) -> (np.ndarray predictions, pd.Index dates)
MODEL_REGISTRY = {
    "AdaBoost": adaboost_model_function,
    "Random Forest": random_forest_model_function,
    "AdaBoost-DI": adaboost_di_model_function,
    "kNN-DI": knn_di_model_function,
    "SVR linear-DI": svr_linear_di_model_function,
}

if HAS_XGB:
    MODEL_REGISTRY["XGBoost-DI"] = xgboost_di_model_function

print(f"Models registered: {list(MODEL_REGISTRY.keys())}")


Models registered: ['arima']


In [20]:

# ============================================================
# Cell 8 — Pipeline Helper
# Fit/transform pipeline tracks before passing to models
# ============================================================

def prepare_track(track_name, fold, target):
    """
    Slice and prepare train/test DataFrames for a given track and fold.

    For DataFrame tracks (A, B): slice directly by date.
    For Pipeline tracks (D, D_mini): slice stationary panel by date,
        fit pipeline on train only, transform both, return DataFrames.

    Parameters
    ----------
    track_name : str — one of ACTIVE_TRACKS
    fold       : dict — fold definition with train/test date boundaries

    Returns
    -------
    train_df : pd.DataFrame
    test_df  : pd.DataFrame
    """
    config = TRACK_CONFIG[track_name][target]

    if config["type"] == "dataframe":
        df = track_data[track_name][target]
        train_df = df.loc[fold["train_start"]:fold["train_end"]]
        test_df  = df.loc[fold["test_start"]:fold["test_end"]]

    elif config["type"] == "pipeline":
        # slice stationary panel
        train_raw = stationary_panel.loc[fold["train_start"]:fold["train_end"]]
        test_raw  = stationary_panel.loc[fold["test_start"]:fold["test_end"]]

        # fit on train only — no leakage
        pipeline = pickle.loads(pickle.dumps(track_data[track_name][target]))  # fresh copy
        pipeline.fit(train_raw)

        # transform both
        train_transformed = pipeline.transform(train_raw)
        test_transformed  = pipeline.transform(test_raw)

        # return as DataFrames preserving date index
        n_cols = train_transformed.shape[1]
        cols   = [f"f{i}" for i in range(n_cols)]

        train_df = pd.DataFrame(train_transformed, index=train_raw.index, columns=cols)
        test_df  = pd.DataFrame(test_transformed,  index=test_raw.index,  columns=cols)

    return train_df, test_df

print("Pipeline helper defined.")

Pipeline helper defined.


In [24]:
# ============================================================
# Cell 9 — Evaluation Loop
# ============================================================

results = []

total_runs = len(folds) * len(ACTIVE_TRACKS) * len(TARGET_LIST) * len(HORIZON_LIST) * len(MODEL_REGISTRY)
completed  = 0

for fold in folds:
    for track_name in ACTIVE_TRACKS:

        for target in TARGET_LIST:
            # slice and prepare track data once per fold/track
            train_df, test_df = prepare_track(track_name, fold, target)

            for horizon in HORIZON_LIST:

                target_col = TARGET_COL_MAP[(target, horizon)]

                # slice target series
                y_train = targets.loc[train_df.index, target_col]
                y_test  = targets.loc[test_df.index,  target_col]

                # NaN guard — skip if target unavailable
                if y_test.isna().any() or y_train.isna().any():
                    continue

                for model_name, model_fn in MODEL_REGISTRY.items():

                    row = {
                        "model":       model_name,
                        "track":       track_name,
                        "target":      target,
                        "horizon":     horizon,
                        "fold_id":     fold["fold_id"],
                        "train_start": fold["train_start"],
                        "train_end":   fold["train_end"],
                        "test_start":  fold["test_start"],
                        "test_end":    fold["test_end"],
                        "error":       None,
                    }

                    try:
            
                        t0 = time.time()
                        y_pred, dates = model_fn(train_df, test_df, y_train)
                        row["runtime_sec"] = round(time.time() - t0, 4)

                        metrics = compute_metrics(y_test.values, y_pred, y_train.values)
                        row.update(metrics)
                    except Exception as e:
                        row["runtime_sec"] = np.nan
                        row["mse"]         = np.nan
                        row["mae"]         = np.nan
                        row["median_ae"]   = np.nan
                        row["mase"]        = np.nan
                        row["rmsse"]       = np.nan
                        row["error"]       = str(e)

                    results.append(row)
                    completed += 1

                    if completed % 1000 == 0:
                        print(f"  {completed}/{total_runs} runs complete")

print(f"\nLoop complete. Total rows: {len(results)}")


KeyboardInterrupt: 

In [22]:
# ============================================================
# Cell 10 — Save Results
# ============================================================

results_df = pd.DataFrame(results)

# --- Save per model ---
for model_name in results_df["model"].unique():
    model_df = results_df[results_df["model"] == model_name]
    out_path = RESULTS_DIR / f"{model_name}_raw.parquet"
    model_df.to_parquet(out_path, index=False)
    print(f"  Saved: {out_path.name}   rows={len(model_df)}")

# --- Summary table ---
summary = (
    results_df
    .groupby(["model", "track", "target", "horizon"])
    [["mse", "mae", "median_ae", "mase", "rmsse"]]
    .mean()
    .round(6)
)

print("\nSummary — mean metrics across folds:")
print(summary.to_string())

  Saved: arima_raw.parquet   rows=3862

Summary — mean metrics across folds:
                                 mse       mae  median_ae  mase     rmsse
model track target horizon                                               
arima B     INDPRO 1        0.000141  0.005682   0.005682   0.0  0.007675
                   3        0.000182  0.006391   0.006391   0.0 -0.660775
                   6        0.000215  0.007919   0.007919   0.0  0.118113
            PAYEMS 1             NaN       NaN        NaN   NaN       NaN
                   3             NaN       NaN        NaN   NaN       NaN
                   6             NaN       NaN        NaN   NaN       NaN


In [23]:
# ============================================================
# Cell 11 — USREC Disaggregation
# Join recession indicator for recession vs expansion analysis
# ============================================================

# rename for clean merge
usrec_col = usrec.columns[0]
usrec_clean = usrec.rename(columns={usrec_col: "usrec"})

# join on test_start date
results_df = results_df.merge(
    usrec_clean,
    left_on="test_start",
    right_index=True,
    how="left"
)

# disaggregated summary
summary_disagg = (
    results_df
    .groupby(["model", "track", "target", "horizon", "usrec"])
    [["mse", "mae", "median_ae", "mase", "rmsse"]]
    .mean()
    .round(6)
)

print("Disaggregated summary — recession (1) vs expansion (0):")
print(summary_disagg.to_string())

Disaggregated summary — recession (1) vs expansion (0):
                                       mse       mae  median_ae  mase     rmsse
model track target horizon usrec                                               
arima B     INDPRO 1       0      0.000053  0.004859   0.004859   0.0  1.274484
                           1      0.000823  0.012040   0.012040   0.0 -9.784413
                   3       0      0.000147  0.005672   0.005672   0.0 -0.464215
                           1      0.000450  0.011930   0.011930   0.0 -2.174823
                   6       0      0.000159  0.006873   0.006873   0.0  0.292077
                           1      0.000641  0.015935   0.015935   0.0 -1.214829
            PAYEMS 1       0           NaN       NaN        NaN   NaN       NaN
                           1           NaN       NaN        NaN   NaN       NaN
                   3       0           NaN       NaN        NaN   NaN       NaN
                           1           NaN       NaN        NaN 